# Enriched calibration: choosing a partner from a few hundred known severe cases

Follows `04_results/enriched_calibration_plan.json`. A site grades (or retrieves from its records) **k severe eyes plus k controls**, and picks a partner for its deployed model from that enriched sample. Same models, sites, eligibility (different family), matched-volume outcome and evaluation (eyes outside the sample) as the random-sample simulation. CPU, about 15 to 20 minutes.

In [1]:
# Cell 1: load
from google.colab import drive
drive.mount('/content/drive')
import os, json, datetime, itertools, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, PREDS, RESULTS, FULL = f'{BASE}/01_splits', f'{BASE}/03_predictions', f'{BASE}/04_results', f'{BASE}/04_results/ballistics_full'
KS = [25, 50, 100, 200]; DRAWS = 200; RNG = np.random.default_rng(20260926)
def pick(df, c): return next(x for x in c if x in df.columns)
def load_scores(path, ids):
    df = pd.read_csv(path, dtype={'image_id': str}); pc = pick(df, ['predicted_probability','probability','prob'])
    s = df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(ids); assert s.notna().all(), path; return s.to_numpy()
# EyePACS: full-recipe zoo
fr = pd.read_csv(f'{SPLITS}/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
Z = pd.DataFrame([('resnet50','resnet'),('resnet50_swsl','resnet'),('resnext50','resnet'),('regnety_032','regnet'),('densenet121','densenet'),
    ('efficientnet_b3','efficientnet'),('effnetv2_s','efficientnet'),('mobilenetv3','mobilenet'),('convnext_t','convnext'),('convnext_t22k','convnext'),
    ('inception_v3','inception'),('vgg16_bn','vgg'),('vit_b','vit'),('vit_b_mae','vit'),('vit_b_dinov2','vit'),('vit_b_clip','vit'),('vit_s','vit'),
    ('deit_s','deit'),('swin_t','swin'),('beitv2_b','beit'),('mixer_b16','mixer'),('retfound','retfound')], columns=['name','family']).set_index('name')
SITES = {'EyePACS': {'ids': fr.image_id.to_numpy(), 'grade': fr[pick(fr, ['level','grade'])].to_numpy().astype(int), 'cluster': fr.patient_id.astype(str).to_numpy(), 'f': 0.25,
                     'family': Z.family.to_dict(), 'S': {m: load_scores(f'{FULL}/preds/{m}_fresh.csv', fr.image_id.to_numpy()) for m in Z.index}}}
dd = pd.read_csv(f'{SPLITS}/ddr_split_intraset_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
DDR_MODELS = {'densenet121_binary':'densenet','densenet121_5class':'densenet','efficientnet_b3_binary':'efficientnet','efficientnet_b3_5class':'efficientnet',
              'resnet50_binary':'resnet','resnet50_5class':'resnet','vit_base_binary':'vit','vit_base_5class':'vit','retfound_binary':'retfound','retfound_5class':'retfound','retfound_adversarial':'retfound'}
A = (dd.half == 'A').to_numpy()
SITES['DDR'] = {'ids': dd.image_id.to_numpy(), 'grade': dd.grade.to_numpy().astype(int), 'cluster': dd.group.astype(str).to_numpy(), 'f': 1.31 * float((dd.grade[A] >= 2).mean()),
                'sample_unit': 'image',   # DDR filename groups are grade-clustered (consecutive numbering), so group sampling gives unrepresentative calibration samples
                'family': DDR_MODELS, 'S': {m: load_scores(f'{PREDS}/ddr_predictions_{m}_v2.csv', dd.image_id.to_numpy()) for m in DDR_MODELS}}
PT = pd.read_csv(f'{FULL}/pair_table.csv'); DEG = {}
for _, r in PT.iterrows(): DEG[(r.a, r.b)] = DEG[(r.b, r.a)] = float(r.sim_degradation)
for k, S in SITES.items(): print(k, len(S['ids']), 'eyes |', len(S['S']), 'models | f =', round(S['f'], 3), '| severe eyes', int((S['grade'] >= 3).sum()))

Mounted at /content/drive
EyePACS 15794 eyes | 22 models | f = 0.25 | severe eyes 712
DDR 12422 eyes | 11 models | f = 0.555 | severe eyes 1149


In [2]:
# Cell 2: per-eye contributions for every ordered pair (thresholds on the full set), strength, consensus
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
for k, S in SITES.items():
    g = S['grade']; sev = g >= 3; M = list(S['S']); S['models'] = M; f = S['f']
    R = {m: refer(S['S'][m], f) for m in M}; RK = {m: rankdata(S['S'][m]) / len(g) for m in M}
    S['net'] = {}
    for a in M:
        for b in M:
            if a == b or S['family'][a] == S['family'][b]: continue
            e = refer(np.maximum(RK[a], RK[b]), f)
            S['net'][(a, b)] = (sev & ~R[a]).astype(np.int8) - (sev & ~e).astype(np.int8)     # per-eye net severe rescue
            S.setdefault('union', {})[(a, b)] = (sev & ~R[a] & R[b])
    S['sevmiss_full'] = {m: int((sev & ~R[m]).sum()) for m in M}; S['R'] = R
    C = np.array([[spearmanr(S['S'][a], S['S'][b])[0] if a != b else np.nan for b in M] for a in M])
    S['consensus'] = dict(zip(M, np.nanmean(C, axis=1)))
    true_strength = [-S['sevmiss_full'][m] for m in M]
    S['consensus_vs_strength'] = round(float(spearmanr([S['consensus'][m] for m in M], true_strength)[0]), 3)
    print(k, 'consensus-vs-true-strength rank correlation:', S['consensus_vs_strength'])

EyePACS consensus-vs-true-strength rank correlation: 0.489
DDR consensus-vs-true-strength rank correlation: 0.264


In [ ]:
# Cell 3: enriched-sample simulation
def eligible(S, a): return [b for b in S['models'] if b != a and S['family'][b] != S['family'][a]]
def draw_enriched(S, k):
    g = S['grade']; sev_idx = np.flatnonzero(g >= 3); non_idx = np.flatnonzero(g < 3)
    m = np.zeros(len(g), bool); m[RNG.choice(sev_idx, k, replace=False)] = True; m[RNG.choice(non_idx, k, replace=False)] = True; return m
def draw_random(S, n):
    m = np.zeros(len(S['grade']), bool); m[RNG.choice(len(m), n, replace=False)] = True; return m
def choose(S, a, rule, m):
    el = eligible(S, a); g = S['grade']; sev_m = (g >= 3) & m
    if rule == 'enriched_rescue':
        best = max(el, key=lambda b: (int(S['net'][(a, b)][sev_m].sum()), int(S['union'][(a, b)][sev_m].sum()), RNG.random())); return best
    if rule in ('enriched_severe_misses', 'random_sample_same_effort'):
        return min(el, key=lambda b: (int((sev_m & ~S['R'][b]).sum()), RNG.random()))
    if rule == 'enriched_auc':
        y = (g[m] >= 3)
        return max(el, key=lambda b: roc_auc_score(y, S['S'][b][m]))
RULES = ['enriched_rescue', 'enriched_severe_misses', 'enriched_auc', 'random_sample_same_effort']
RES = {}
for k_, S in SITES.items():
    ts = {a: min(eligible(S, a), key=lambda b: S['sevmiss_full'][b]) for a in S['models']}
    out = {k: {r: [] for r in RULES + ['random', 'true_strongest', 'oracle']} for k in KS}
    hits = {k: {r: [] for r in RULES} for k in KS}
    nsev = int((S['grade'] >= 3).sum())
    for k in KS:
        if k > nsev // 2: print(k_, 'skip k =', k, '(too few severe eyes)'); continue
        for d in range(DRAWS):
            me = draw_enriched(S, k); mr = draw_random(S, 2 * k)
            for r in RULES + ['random', 'true_strongest', 'oracle']:
                m = mr if r == 'random_sample_same_effort' else me; ev = ~m; vals = []
                for a in S['models']:
                    el = eligible(S, a); score = {b: int(S['net'][(a, b)][ev].sum()) for b in el}
                    if r == 'random': vals.append(np.mean(list(score.values())))
                    elif r == 'oracle': vals.append(max(score.values()))
                    elif r == 'true_strongest': vals.append(score[ts[a]])
                    else:
                        b = choose(S, a, r, m); vals.append(score[b]); hits[k][r].append(b == ts[a])
                out[k][r].append(float(np.mean(vals)))
        print(k_, 'k =', k, {r: round(float(np.mean(v)), 2) for r, v in out[k].items() if v})
    RES[k_] = {'out': out, 'hits': hits, 'n_severe': nsev}

EyePACS k = 25 {'enriched_rescue': 12.67, 'enriched_severe_misses': 13.22, 'enriched_auc': 14.32, 'random_sample_same_effort': 11.86, 'random': 11.17, 'true_strongest': 17.37, 'oracle': 19.65}
EyePACS k = 50 {'enriched_rescue': 13.01, 'enriched_severe_misses': 13.52, 'enriched_auc': 14.17, 'random_sample_same_effort': 12.11, 'random': 10.77, 'true_strongest': 16.73, 'oracle': 19.02}
EyePACS k = 100 {'enriched_rescue': 12.73, 'enriched_severe_misses': 13.33, 'enriched_auc': 14.03, 'random_sample_same_effort': 12.34, 'random': 9.95, 'true_strongest': 15.37, 'oracle': 17.7}
EyePACS k = 200 {'enriched_rescue': 11.25, 'enriched_severe_misses': 11.59, 'enriched_auc': 12.12, 'random_sample_same_effort': 12.68, 'random': 8.3, 'true_strongest': 12.85, 'oracle': 15.11}
DDR k = 25 {'enriched_rescue': 15.92, 'enriched_severe_misses': 16.61, 'enriched_auc': 18.91, 'random_sample_same_effort': 15.3, 'random': 14.67, 'true_strongest': 23.14, 'oracle': 23.97}
DDR k = 50 {'enriched_rescue': 16.44, 'enr

In [ ]:
# Cell 4: summaries, figure, write
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
SUM = {}; fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
COL = {'enriched_rescue': '#D55E00', 'enriched_severe_misses': '#0072B2', 'enriched_auc': '#56B4E9', 'random_sample_same_effort': '#CC79A7'}
for ax, (k_, R_) in zip(axes, RES.items()):
    out = R_['out']; SUM[k_] = {}; ks = [k for k in KS if out[k]['random']]
    for k in ks:
        rnd, ts, orc = np.mean(out[k]['random']), np.mean(out[k]['true_strongest']), np.mean(out[k]['oracle'])
        SUM[k_][k] = {r: {'mean_net_severe_rescued': round(float(np.mean(v)), 2), 'p5': round(float(np.percentile(v, 5)), 2), 'p95': round(float(np.percentile(v, 95)), 2),
                          'share_of_true_strongest_advantage': round(float((np.mean(v) - rnd) / (ts - rnd)), 3),
                          'share_of_oracle_advantage': round(float((np.mean(v) - rnd) / (orc - rnd)), 3),
                          **({'picks_true_strongest': round(float(np.mean(R_['hits'][k][r])), 3)} if r in RULES else {})}
                      for r, v in out[k].items()}
    for r, c in COL.items():
        y = [SUM[k_][k][r]['share_of_true_strongest_advantage'] for k in ks]
        ax.plot([2 * k for k in ks], y, '-o', color=c, lw=2, label=r.replace('_', ' '))
    ax.axhline(1, color='#009E73', ls='--', lw=1, label='true strongest'); ax.axhline(0, color='#999', ls='--', lw=1, label='random')
    orc_share = [SUM[k_][k]['oracle']['share_of_true_strongest_advantage'] for k in ks]
    ax.plot([2 * k for k in ks], orc_share, ':', color='k', lw=1.5, label='oracle')
    ax.set_xscale('log'); ax.set_xlabel('Graded images (k severe + k controls; random sample of same size for reference)')
    ax.set_ylabel('Share of the true-strongest benefit over random'); ax.set_title(f"{k_} ({R_['n_severe']} severe eyes available)", loc='left', fontsize=10, weight='bold')
    ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(fontsize=7.5, frameon=False)
fig.suptitle('Choosing a partner from an enriched local sample of severe cases', x=0.01, ha='left', fontsize=12, weight='bold'); fig.tight_layout()
fig.savefig(f'{BASE}/05_figures/enriched_calibration_v2.png', dpi=220, bbox_inches='tight'); plt.close(fig)
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'enriched_calibration_plan.json', 'status': 'exploratory', 'summary': SUM,
       'n_severe': {k: v['n_severe'] for k, v in RES.items()}}
json.dump(res, open(f'{RESULTS}/enriched_calibration_results.json', 'w'), indent=2, default=str)
for k_ in SUM: print(k_); print(pd.DataFrame({k: {r: v['share_of_true_strongest_advantage'] for r, v in SUM[k_][k].items()} for k in SUM[k_]}).to_string())
print('Wrote 04_results/enriched_calibration_results.json and 05_figures/enriched_calibration_v2.png')

EyePACS
                             25     50     100    200
enriched_rescue            0.242  0.375  0.513  0.648
enriched_severe_misses     0.330  0.462  0.624  0.722
enriched_auc               0.507  0.570  0.753  0.840
random_sample_same_effort  0.111  0.226  0.442  0.963
random                     0.000  0.000  0.000  0.000
true_strongest             1.000  1.000  1.000  1.000
oracle                     1.369  1.383  1.430  1.497
DDR
                             25     50     100    200
enriched_rescue            0.147  0.255  0.349  0.533
enriched_severe_misses     0.229  0.370  0.481  0.648
enriched_auc               0.500  0.626  0.767  0.821
random_sample_same_effort  0.074  0.133  0.356  0.718
random                     0.000  0.000  0.000  0.000
true_strongest             1.000  1.000  1.000  1.000
oracle                     1.098  1.102  1.111  1.125
Wrote 04_results/enriched_calibration_results.json and 05_figures/enriched_calibration_v2.png
